# Model selection: LFM3-1.7B LoRA on the realistic data

The production fine-tuning script (`ml/finetune/train_lora.py`, `config-lfm3.yaml`) on the same frozen train split, rendered with the production prompt (v2, full style, as in the evaluation). One run per seed in SEEDS; each exported to GGUF. LFM Open License v1.0.

In [ ]:
# Setup: the experiment branch (public), dependencies, the frozen split
import os, subprocess, hashlib, json
if not os.path.exists("/tmp/codeply"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", "codeply-li-dtw-experiment",
                    "https://github.com/AwaisSDev/Codeply-Craft", "/tmp/codeply"], check=True)
os.chdir("/tmp/codeply/research/dtw")  # outside /kaggle/working: the output keeps only artifacts
subprocess.run("pip -q install 'transformers>=4.48' peft accelerate sentencepiece pyyaml", shell=True, check=True)
subprocess.run("cd data/real && for f in *.gz; do gunzip -kf $f; done", shell=True, check=True)
split = json.load(open("data/real/SPLIT.json"))
assert split["leakChecksPass"], "leak checks failed: do not train"
for k in ("train", "val", "test"):
    assert hashlib.sha256(open(f"data/real/{k}.jsonl", "rb").read()).hexdigest() == split["hashes"][k], f"{k} changed"
print({k: v["n"] for k, v in split["splits"].items()})
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# llama.cpp at the pinned commit: GGUF converter and quantizer
import os, subprocess
if not os.path.exists("/tmp/llama.cpp"):
    subprocess.run("git clone https://github.com/ggml-org/llama.cpp /tmp/llama.cpp && cd /tmp/llama.cpp && git checkout aa588a3463357ee990b314f41dda242869130704", shell=True, check=True)
    subprocess.run("cd /tmp/llama.cpp && cmake -B build -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF > /dev/null && cmake --build build -j4 --target llama-quantize > /dev/null", shell=True, check=True)
    subprocess.run("pip -q install -e /tmp/llama.cpp/gguf-py", shell=True, check=True)
def export_gguf(run):
    subprocess.run(f"python /tmp/llama.cpp/convert_hf_to_gguf.py {run}/backbone --outtype f16 --outfile {run}/backbone-f16.gguf", shell=True, check=True)
    for q in ("Q8_0", "Q4_0"):
        subprocess.run(f"/tmp/llama.cpp/build/bin/llama-quantize {run}/backbone-f16.gguf {run}/backbone-{q.lower()}.gguf {q}", shell=True, check=True)
    subprocess.run(f"python model/export_heads.py {run}", shell=True, check=True)

In [ ]:
import subprocess, os, yaml, time
SEEDS = [1, 2]   # one per T4 (the kernel has two); the protocol asks for 3 seeds in total
W = "/kaggle/working/lora"   # under /kaggle/working so that a run stopped by the 12 h limit keeps its checkpoints
os.chdir("/tmp/codeply")
subprocess.run("pip -q install -r ml/finetune/requirements.txt", shell=True, check=True)
# Kaggle's image ships torchao 0.10; PEFT refuses any torchao below 0.16 (the first run failed on it). LoRA does not use it.
subprocess.run("pip -q uninstall -y torchao", shell=True, check=True)
subprocess.run("node --version || (curl -fsSL https://deb.nodesource.com/setup_20.x | bash - && apt-get install -y nodejs) > /dev/null", shell=True, check=True)
subprocess.run("cp research/dtw/data/real/train.jsonl research/dtw/data/real/val.jsonl /tmp/ && node ml/data/export-sft.mjs --in /tmp --style full", shell=True, check=True)
# export_gguf.sh calibrates the INT4 importance matrix on ml/data/out/sft-val.jsonl: give it the realistic val prompts.
subprocess.run("cp /tmp/sft-val.jsonl ml/data/out/sft-val.jsonl", shell=True, check=True)
procs = {}
for gpu, seed in enumerate(SEEDS):
    cfg = yaml.safe_load(open("ml/finetune/config-lfm3.yaml"))
    # Same effective batch as config-lfm3.yaml (16), micro-batch 1 for the T4's 16 GB: with batch 4 the
    # backward pass ran out of memory on the 128K-vocabulary logits (run 2).
    cfg["grad_accum"] = cfg["grad_accum"] * cfg["batch_size"]
    cfg["batch_size"] = 1
    out = f"{W}/real-lfm3-lora-s{seed}"
    cfg.update({"train_file": "/tmp/sft-train.jsonl", "val_file": "/tmp/sft-val.jsonl", "seed": seed, "output_dir": out})
    os.makedirs(out, exist_ok=True)
    yaml.safe_dump(cfg, open(f"/tmp/config-s{seed}.yaml", "w"))
    if not os.path.exists(f"{out}/merged"):
        # Both seeds at once, one GPU each (run 3 used one of the two T4s and needed ~17.6 h per seed).
        procs[seed] = subprocess.Popen(f"CUDA_VISIBLE_DEVICES={gpu} PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True "
                                       f"python ml/finetune/train_lora.py --config /tmp/config-s{seed}.yaml > {out}/stdout.log 2>&1", shell=True)
while any(p.poll() is None for p in procs.values()):
    time.sleep(600)
    for seed in procs:
        print(seed, open(f"{W}/real-lfm3-lora-s{seed}/stdout.log").read().strip().splitlines()[-1][:200], flush=True)
for seed, p in procs.items():
    print(f"seed {seed} exit {p.returncode}:", open(f"{W}/real-lfm3-lora-s{seed}/stdout.log").read()[-3000:], flush=True)

In [ ]:
!cd /tmp/llama.cpp && cmake --build build -j4 --target llama-imatrix > /dev/null
import subprocess, shutil, glob, os
for seed in SEEDS:
    out = f"{W}/real-lfm3-lora-s{seed}"
    if not os.path.exists(f"{out}/merged"):
        print(f"seed {seed}: no merged model (training did not finish); adapters:", glob.glob(f"{out}/adapter*")); continue
    # f16 GGUF and imatrix in /tmp (not kept); the two quantised models go to the output; the merged weights are
    # deleted once exported (the adapter, ~130 MB, stays and can be merged again).
    subprocess.run(f"cd /tmp/codeply && OUT=/tmp/gguf-s{seed} LLAMA_CPP=/tmp/llama.cpp bash ml/finetune/export_gguf.sh {out}/merged real-lfm3-lora-s{seed}", shell=True, check=True)
    for f in glob.glob(f"/tmp/gguf-s{seed}/*Q4_K_M.gguf") + glob.glob(f"/tmp/gguf-s{seed}/*Q8_0.gguf"):
        shutil.copy(f, out)
    shutil.rmtree(f"{out}/merged")
    shutil.rmtree(f"/tmp/gguf-s{seed}")

In [ ]:
!ls -laR /kaggle/working/lora | grep -v "^total"